# Sort the copied `public` tables by series

Run jobs read in **shards**: ranges of gauges, each with its linked forecasts. A shard
skips every data file whose `series_key` range (from the file's column statistics) lies
outside it, so the fewer series each file spans, the less each shard reads. Scale tests
showed `public.nwm30_medium_range` only partly sorted: each
of 144 shards read about 5 of January's 22 files (an overlap of 33; 1 is ideal).

This rewrites the tables' files sorted by the tables' declared sort order (`series_key`,
then time), with Iceberg's `rewrite_data_files` procedure (`strategy => 'sort'`), a month
partition at a time. Each partition is sorted as a whole, so its files hold disjoint
`series_key` ranges.

- **Rows are unchanged:** a rewrite replaces files with the same rows. Readers at the old
  snapshot (pinned runs) still see their files: nothing is expired here.
- **Storage:** the old files stay until snapshots are expired (not done here), so the
  table's storage about doubles meanwhile.
- **Re-runnable:** each month is rewritten whole every time (`rewrite-all`); pick months
  in `MONTHS`.

Start with **2. Layout**: it writes nothing. Then rewrite January 2024 alone, time a month's
run against it, and compare before doing the rest.

In [ ]:
import pandas as pd
from teehr.evaluation.spark_session_utils import create_spark_session

## 1. Settings

In [ ]:
# table: (time column it's partitioned by month of, sort order)
TABLES = {
    "nwm30_medium_range": ("reference_time", "series_key, reference_time, value_time"),
    # Observations were less spread (overlap ~7); add them if the forecasts pay off:
    # "usgs_streamflow": ("value_time", "series_key, value_time"),
}
PUBLIC = "iceberg.public"
MONTHS = ["2024-01"]  # months (YYYY-MM) to rewrite; None = every month of the table
TARGET_FILE_BYTES = 512 * 1024**2  # Iceberg's default

In [ ]:
# A month is ~250M rows (~11 GB of Parquet) of medium range: sorting one wants a cluster.
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=8,
    executor_memory="16g",
    executor_cores=4,
)
spark.conf.set("spark.sql.session.timeZone", "UTC")  # months are UTC (Iceberg's are)

## 2. Layout (writes nothing)

Per month, from the table's `files` metadata (no data read): the files, rows, size, and
how many files' `series_key` ranges each file's range overlaps (itself included).
`overlap` 1 means files hold disjoint ranges of series; the file count means every
file spans every series, so every shard reads all of them.

In [ ]:
def month_of(value):
    """YYYY-MM of an Iceberg month partition value (months since 1970-01)."""
    return f"{1970 + value // 12:04d}-{value % 12 + 1:02d}"


def layout(table):
    time_column, _ = TABLES[table]
    files = spark.sql(f"""
        SELECT partition.{time_column}_month AS m, record_count, file_size_in_bytes,
               readable_metrics.series_key.lower_bound AS lo,
               readable_metrics.series_key.upper_bound AS hi
        FROM {PUBLIC}.{table}.files
    """).toPandas()
    files["month"] = files.m.map(month_of)
    rows = []
    for month, f in files.groupby("month"):
        lo, hi = f.lo.to_numpy(), f.hi.to_numpy()
        overlaps = [((lo <= h) & (hi >= l)).sum() for l, h in zip(lo, hi)]
        rows.append({
            "table": table, "month": month, "files": len(f),
            "rows": int(f.record_count.sum()),
            "gb": round(f.file_size_in_bytes.sum() / 1e9, 2),
            "overlap": round(sum(overlaps) / len(f), 1),
        })
    return pd.DataFrame(rows)


before = pd.concat([layout(t) for t in TABLES])
print(before.groupby("table")[["files", "rows", "gb"]].sum())
before[before.month.isin(MONTHS)] if MONTHS else before

## 3. Rewrite, sorted

One month partition per call. Each call reports the files it replaced and added. A
month of medium range takes a few minutes on the cluster above.

In [ ]:
%%time
def month_range(column, month):
    start = pd.Period(month, freq="M")
    return (f"{column} >= TIMESTAMP '{start.start_time:%Y-%m-%d} 00:00:00' AND "
            f"{column} < TIMESTAMP '{(start + 1).start_time:%Y-%m-%d} 00:00:00'")


results = []
for table, (time_column, sort_order) in TABLES.items():
    months = MONTHS or sorted(before[before.table == table].month)
    for month in months:
        r = spark.sql(f"""
            CALL iceberg.system.rewrite_data_files(
                table => 'public.{table}',
                strategy => 'sort',
                sort_order => '{sort_order}',
                where => "{month_range(time_column, month)}",
                options => map(
                    'rewrite-all', 'true',
                    'target-file-size-bytes', '{TARGET_FILE_BYTES}',
                    'partial-progress.enabled', 'true'
                )
            )
        """).first().asDict()
        results.append({"table": table, "month": month, **r})
        print(f"{table} {month}: {r['rewritten_data_files_count']} files -> "
              f"{r['added_data_files_count']}", flush=True)
pd.DataFrame(results)

## 4. Layout after

Same rows per month as before; `overlap` should now be close to 1.

In [ ]:
after = pd.concat([layout(t) for t in TABLES])
both = before.merge(after, on=["table", "month"], suffixes=("_before", "_after"))
assert (both.rows_before == both.rows_after).all(), "row counts changed"
cols = ["table", "month", "files_before", "files_after", "overlap_before",
        "overlap_after", "gb_before", "gb_after"]
(both[both.month.isin(MONTHS)] if MONTHS else both)[cols]

## 5. Next

Re-run a month's evaluation over the sorted month and compare
`file_reads` and run time with the unsorted run: 735 file reads, 9.5 minutes. If it pays
off, set `MONTHS = None` here and rewrite the rest.

In [ ]:
spark.stop()